In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree

# 1. CARGAR MAESTRO DE COORDENADAS
ficheros_coords = [
    '202468-7-intensidad-trafico-xlsx.xlsx',
    '202468-13-intensidad-trafico-xlsx.xlsx',
    '202468-14-intensidad-trafico-xlsx.xlsx',
    '202468-15-intensidad-trafico-xlsx.xlsx',
    '202468-16-intensidad-trafico-xlsx.xlsx',
    '202468-18-intensidad-trafico-xlsx.xlsx',
    '202468-235-intensidad-trafico-xlsx.xlsx',
    '202468-240-intensidad-trafico-xlsx.xlsx'
]

def obtener_maestro_sensores(lista):
    dfs = []
    for f in lista:
        df = pd.read_excel(f).rename(columns=lambda x: x.strip())
        if 'idelem' in df.columns: df = df.rename(columns={'idelem': 'id'})
        col_x = 'st_x' if 'st_x' in df.columns else 'utm_x'
        col_y = 'st_y' if 'st_y' in df.columns else 'utm_y'
        if 'id' in df.columns and col_x in df.columns:
            dfs.append(df[['id', col_x, col_y]].rename(columns={col_x: 'sensor_coordenada_x_utm', col_y: 'sensor_coordenada_y_utm'}))
    return pd.concat(dfs).drop_duplicates('id').dropna()

print("Cargando maestro de sensores...")
df_sensores = obtener_maestro_sensores(ficheros_coords)
df_sensores.to_csv('sensores_con_coordenadas.csv', index=False, encoding='latin-1')

# 2. ENCONTRAR SENSOR MÁS CERCANO PARA CADA ACCIDENTE
print("Encontrando sensor más cercano para cada accidente...")
df_acc = pd.read_csv('madrid_accidentes_limpio_festivos.csv', encoding='latin-1')
df_acc = df_acc.dropna(subset=['coordenada_x_utm', 'coordenada_y_utm']).copy()

df_acc['fecha_completa'] = df_acc['fecha'].astype(str) + " " + df_acc['hora'].astype(str)

tree = KDTree(df_sensores[['sensor_coordenada_x_utm', 'sensor_coordenada_y_utm']].values)
dist, indices = tree.query(df_acc[['coordenada_x_utm', 'coordenada_y_utm']].values)

df_acc['id_sensor_cercano'] = df_sensores.iloc[indices]['id'].values
ids_necesarios = set(df_acc['id_sensor_cercano'].unique())
fechas_necesarias = set(df_acc['fecha_completa'].unique())

# 3. LEER TRÁFICO GIGANTE POR TROZOS (CHUNKS)
# Solo guardamos las filas cuyo 'id' esté en nuestra lista de accidentes
print("Procesando archivo de tráfico gigante por trozos...")
chunk_size = 100000 
lista_pedazos_trafico = []

# Iteramos sobre el CSV gigante sin cargarlo todo en RAM
for chunk in pd.read_csv('trafico_madrid_acumulado.csv', encoding='latin-1', sep=';', chunksize=chunk_size):
    chunk.columns = chunk.columns.str.strip()
    pedazo_util = chunk[(chunk['id'].isin(ids_necesarios)) & (chunk['fecha'].isin(fechas_necesarias))]
    if not pedazo_util.empty:
        lista_pedazos_trafico.append(pedazo_util)

df_trafico_filtrado = pd.concat(lista_pedazos_trafico)

Cargando maestro de sensores...
Encontrando sensor más cercano para cada accidente...
Procesando archivo de tráfico gigante por trozos...
Uniendo accidentes con datos de tráfico...


KeyError: 'fecha_completa'

In [ ]:
print("Uniendo accidentes con datos de tráfico...")
df_final = pd.merge(
    df_acc, 
    df_trafico_filtrado, 
    left_on=['id_sensor_cercano', 'fecha_completa'],
    right_on=['id', 'fecha'],
    how='left'
)

Uniendo accidentes con datos de tráfico...


KeyError: "['id_y'] not found in axis"

In [8]:
df_final = df_final.drop(columns=['fecha_y', 'id']).rename(columns={'fecha_x': 'fecha'})
# 5. GUARDAR
print("Guardando resultado final...")
df_final.to_csv('accidentes_con_trafico_final.csv', index=False, encoding='latin-1')
print("Completado.")

Guardando resultado final...
Completado.
